# Warm-up: Conditionals Light Board

Run the setup cell first. Then move the sliders and type in the box: the light shows **which branch of the code ran**.

`light()` works just like `print()`: give it a color, then the message.

The `%%slider` or `%%textbox` line at the top of a cell just adds the control; the code underneath is a normal `if` statement.

In [ ]:
# @title ▶ Run this first (sets up the lights)
from IPython.display import display, HTML, clear_output
from IPython.core.magic import register_cell_magic
import ipywidgets as widgets


def light(color, *message):
    """Show a colored indicator light with a message next to it."""
    text = ' '.join(str(m) for m in message)
    html = ('<div style="display:flex;align-items:center;gap:18px;'
            'font-family:sans-serif;font-size:22px;padding:6px 0">'
            '<div style="width:56px;height:56px;border-radius:50%;'
            'background:' + color + ';box-shadow:0 0 24px ' + color + ';'
            'border:3px solid #333"></div>'
            '<b>' + text + '</b></div>')
    display(HTML(html))


live_controls = {}


def run_live(cell, controls):
    """Show the controls, and re-run the cell's code every time one changes."""
    live_controls.clear()
    live_controls.update(controls)
    out = widgets.Output()
    namespace = get_ipython().user_ns

    def update(change=None):
        for name in controls:
            namespace[name] = controls[name].value
        with out:
            clear_output(wait=True)
            exec(cell, namespace)

    for control in controls.values():
        control.observe(update, names='value')
    display(widgets.HBox(list(controls.values())), out)
    update()


@register_cell_magic
def slider(line, cell):
    """%%slider name low high start [step]"""
    parts = line.split()
    name = parts[0]
    numbers = parts[1:]
    whole = True
    for n in numbers:
        if '.' in n:
            whole = False
    low = float(numbers[0])
    high = float(numbers[1])
    start = float(numbers[2]) if len(numbers) > 2 else low
    if len(numbers) > 3:
        step = float(numbers[3])
    elif whole:
        step = max(1, (high - low) // 100)
    else:
        step = 0.1
    layout = widgets.Layout(width='500px')
    style = {'description_width': 'initial'}
    if whole:
        control = widgets.IntSlider(value=int(start), min=int(low), max=int(high),
                                    step=int(step), description=name,
                                    layout=layout, style=style)
    else:
        control = widgets.FloatSlider(value=start, min=low, max=high, step=step,
                                      description=name, layout=layout, style=style)
    run_live(cell, {name: control})


@register_cell_magic
def textbox(line, cell):
    """%%textbox name"""
    name = line.split()[0]
    control = widgets.Text(value='', description=name, placeholder='type here',
                           continuous_update=True,
                           style={'description_width': 'initial'})
    run_live(cell, {name: control})


print('Lights ready!')


## 1. The radon traffic light

Move the slider. Watch which branch of the `if / elif / else` lights up.

**Try it:** what happens at exactly 4.0? At exactly 2.0? Why?

In [ ]:
%%slider reading 0.0 8.0 3.0
if reading > 4:
    light('red', 'POOR: action required')
elif reading >= 2:
    light('gold', 'FAIR')
else:
    light('green', 'GOOD')

## 2. Concrete cylinder: PASS or FAIL?

Same 6-inch cylinder from Lesson 1. Slide the load up.

**Try it:** find the load where the light flips from PASS to FAIL.

In [ ]:
%%slider load 50000 150000 90000
area = 28.27          # sq in (6-inch diameter cylinder)
design_limit = 4000   # psi
stress = load / area

if stress > design_limit:
    light('red', 'FAIL: stress =', stress, 'psi')
else:
    light('limegreen', 'PASS: stress =', stress, 'psi')

## 3. Spot the bug: elif order

This code is supposed to show LOW, MODERATE, or HIGH stress. Slide all the way up to 6000 psi.

**Try it:** why does HIGH never light up? (Remember: Python checks each condition **in order** and stops at the first `True`.) Fix the order and run the cell again.

In [ ]:
%%slider stress 0 6000 1000 100
if stress >= 2000:
    light('gold', 'MODERATE: acceptable, monitor')
elif stress > 4000:
    light('red', 'HIGH: exceeds design limit')
else:
    light('green', 'LOW: well within limit')

## 4. `==` is picky

Type a building type in the box. The light only turns green for an exact match.

**Try it:** type `Office`. Then try `office`, `OFFICE`, and `Office ` (with a space at the end).

In [ ]:
%%textbox building_type
if building_type == 'Office':
    light('limegreen', 'Match!')
else:
    light('red', 'No match')

## 5. A row of lights: `for` + `if`

The loop checks every radon reading and turns on one light per reading.

**Try it:** add your own readings to the list and run the cell again.

In [ ]:
readings = [1.2, 3.5, 5.1, 0.8, 4.2, 2.0]   # pCi/L

for reading in readings:
    if reading > 4:
        light('red', reading, 'POOR')
    elif reading >= 2:
        light('gold', reading, 'FAIR')
    else:
        light('green', reading, 'GOOD')